In [ ]:
from pathlib import Path
import os
import sys

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
sys.path.insert(0, str(project_root))
from experiments.paths import GENERATED, PROJECT_ROOT

run_directory = GENERATED / 'soay/age_size/validation/covariance/year2'
run_directory.mkdir(parents=True, exist_ok=True)
os.chdir(run_directory)


# Age–Size DeepRV Covariance Evaluation

Set `result_directory` to the downloaded HPC job directory, then run both code cells.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

result_directory = GENERATED / 'soay/age_size/validation/covariance/year2'
result_path = result_directory / "results" / "dowson_landau_age_size_by_parameter.csv"
figure_directory = result_directory / "figures"
figure_directory.mkdir(parents=True, exist_ok=True)

if not result_path.exists():
    raise FileNotFoundError(f"Set result_directory to the downloaded HPC job directory. Missing: {result_path}")

results = pd.read_csv(result_path)
vital_rate_order = ["Survival", "Growth", "Reproduction"]
colors = {"Survival": "#0072B2", "Growth": "#D55E00", "Reproduction": "#009E73"}
required_columns = {"Vital Rate", "Parameter Draw", "Size Lengthscale", "Age Lengthscale", "Relative D Covariance Squared"}
missing_columns = sorted(required_columns - set(results.columns))
if missing_columns:
    raise ValueError(f"Missing result columns: {missing_columns}")
for vital_rate in vital_rate_order:
    if len(results[results["Vital Rate"] == vital_rate]) != 5000:
        raise ValueError(f"{vital_rate} does not contain exactly 5,000 completed parameter draws")

print(f"Loaded {len(results):,} Completed Covariance Evaluations")

In [ ]:
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

distance_column = "Relative D Covariance Squared"
distance_threshold = results[distance_column].quantile(1.0)
plot_results = results[results[distance_column] <= distance_threshold]
figure, axes = plt.subplots(3, 2, figsize=(5.906, 5.2), sharey=True, constrained_layout=True)

for row, vital_rate in enumerate(vital_rate_order):
    current = plot_results[plot_results["Vital Rate"] == vital_rate]
    for column, lengthscale_name in enumerate(["Size Lengthscale", "Age Lengthscale"]):
        axis = axes[row, column]
        axis.scatter(current[lengthscale_name], current[distance_column], color=colors[vital_rate], s=5, alpha=0.35, edgecolors="none")
        axis.grid(False)
        axis.tick_params(direction="out")
        axis.spines["top"].set_visible(False)
        axis.spines["right"].set_visible(False)
    axes[row, 1].annotate(vital_rate, xy=(1.04, 0.5), xycoords="axes fraction", va="center", fontsize=9)

axes[0, 0].set_title("Size Lengthscale")
axes[0, 1].set_title("Age Lengthscale")
axes[-1, 0].set_xlabel("Size Lengthscale")
axes[-1, 1].set_xlabel("Age Lengthscale")
axes[0, 0].set_ylim(0, distance_threshold * 1.02)
figure.supylabel("Relative Squared Covariance Distance")
figure.savefig(figure_directory / "age_size_relative_squared_covariance_distance.pdf")
figure.savefig(figure_directory / "age_size_relative_squared_covariance_distance.png", dpi=300)
plt.show()

In [ ]:
from matplotlib.colors import Normalize

plt.style.use("seaborn-v0_8-white")
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

distance_column = "Relative D Covariance Squared"
distance_threshold = results[distance_column].quantile(1.0)
colour_scale = Normalize(vmin=0, vmax=distance_threshold)

figure, axes = plt.subplots(1, 3, figsize=(5.906, 2.4), constrained_layout=True)

for axis, vital_rate in zip(axes, vital_rate_order):
    current = results[results["Vital Rate"] == vital_rate]

    points = axis.scatter(
        current["Size Lengthscale"],
        current["Age Lengthscale"],
        c=current[distance_column],
        cmap="viridis",
        norm=colour_scale,
        s=10,
        alpha=0.8,
        edgecolors="none",
    )

    axis.set_title(vital_rate)
    axis.set_xlabel("Size Lengthscale", fontsize=10)
    axis.set_ylabel("Age Lengthscale", fontsize=10)
    axis.grid(False)
    axis.tick_params(direction="out")
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)

colourbar = figure.colorbar(points, ax=axes, pad=0.02)
colourbar.set_label(r"$\tilde{D}_{\mathrm{DL}}^2$")
colourbar.ax.tick_params(labelsize=8)

figure.savefig(figure_directory / "age_size_joint_lengthscales_covariance_distance.pdf")
figure.savefig(figure_directory / "age_size_joint_lengthscales_covariance_distance.png", dpi=300)

plt.show()